# MP2: OSS activity analysis — bdowlin2

Part 1: WoC retrieval and source validation. See `bdowlin2_vis.ipynb` for monthly activity and gap analysis.

## Sources and method

Historical commit details are strictly from WoC. All 83,535 successfully retrieved records are analyzed across the ten assigned projects. The TA approved analysis using reduced records; the user selected omission of the 54 unavailable WoC objects (OpenDA 7, GCAM 11, OpenMS 36). Their exact hashes and errors are documented in `bdowlin2_unavailable_woc_commits.csv` and `WOC_DATA_EXCEPTIONS.md`.

All historical metrics describe the retrieved records. Missing objects are excluded explicitly, never zero-filled or replaced with GitHub details. The seven unaffected projects have complete manifest coverage. `bdowlin2_collection_coverage.csv` reports expected, analyzed, and unavailable counts as proof of collection; seven projects exceed 1,000 analyzed records, and the other three have all their manifest records.

GitHub supplies stars/forks, the latest eligible commit and recent themes, and supplemental evidence. Status uses September 30, 2026 with April 1, 2026 as the six-month threshold, as selected by the user. Stars/forks are retrieval-time values. The live WoC data includes some 2026 commits; no uniform January 2025 snapshot is assumed. Counts span each project's first through last observed commit month, without trailing zero months.

In [1]:
"""Resumable retrieval from WoC's public HTTP API (Python client)."""
from pathlib import Path
import csv,json,time,urllib.request,urllib.parse,urllib.error,sqlite3,os

ROOT=Path(os.environ.get('MP2_ROOT',Path.cwd()))
NETID='bdowlin2'
CACHE=ROOT/'work'/'cache'
CACHE.mkdir(parents=True,exist_ok=True)
FIELDS=['project_wocid','commit_sha1','author','time','commit message']

def get_json(url, attempts=3):
    for attempt in range(attempts):
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'CS445-MP2-research','Accept':'application/json'})
            with urllib.request.urlopen(req,timeout=30) as r: return json.load(r)
        except Exception as e:
            if isinstance(e,urllib.error.HTTPError) and e.code in (400,404,422): raise
            if attempt==attempts-1: raise
            delay=max(5*2**attempt,int(e.headers.get('Retry-After','0')) if isinstance(e,urllib.error.HTTPError) else 0)
            print(f'Retry {attempt+1}: {type(e).__name__}; waiting {delay}s',flush=True)
            time.sleep(delay)

def projects():
    with (ROOT/'net2prj.csv').open(encoding='utf-8-sig',newline='') as f:
        return [r for r in csv.DictReader(f) if r['netID']==NETID]

def get_shas():
    result={}
    for p in projects():
        path=CACHE/(p['WoC']+'_shas.json')
        if path.exists(): shas=json.loads(path.read_text())
        else:
            shas=[]; cursor=0
            while cursor is not None:
                d=get_json('https://worldofcode.org/api/lookup/map/p2c/'+p['WoC']+'?cursor='+str(cursor))
                shas.extend(d['data']); cursor=d.get('nextCursor'); time.sleep(.5)
            shas=sorted(set(shas)); path.write_text(json.dumps(shas))
        result[p['WoC']]=shas
        print(p['WoC'],len(shas),'SHAs',flush=True)
    with (ROOT/(NETID+'_commit_shas.csv')).open('w',encoding='utf-8',newline='') as f:
        w=csv.writer(f); w.writerow(FIELDS[:2]); w.writerows((p,s) for p,ss in result.items() for s in ss)
    return result

def fetch_batch(keys):
    # WoC's endpoint accepts at most 10 keys per request.
    u='https://worldofcode.org/api/lookup/object/commit?'+urllib.parse.urlencode([('q',k) for k in keys])
    return get_json(u)

def retrieve():
    shas=get_shas()
    db=sqlite3.connect(CACHE/'commits.sqlite')
    db.execute('CREATE TABLE IF NOT EXISTS commits (sha TEXT PRIMARY KEY, author TEXT, time INTEGER, message TEXT)')
    done={r[0] for r in db.execute('SELECT sha FROM commits')}
    # Finish smaller projects first, while keeping each SHA unique.
    ordered=dict.fromkeys(s for p in sorted(shas,key=lambda p:len(shas[p])) for s in shas[p])
    missing=[s for s in ordered if s not in done]
    print('Need details:',len(missing),'Cached:',len(done),flush=True)
    failures={}; started=time.monotonic()
    consecutive_failures=0
    for start in range(0,len(missing),10):
        batch=missing[start:start+10]
        try:
            d=fetch_batch(batch)
            failures.update(d.get('errors',{}))
            for sha,c in d['data'].items():
                assert sha in batch and len(c)==5 and len(c[2])==3,(sha,c)
                db.execute('INSERT OR REPLACE INTO commits VALUES (?,?,?,?)',(sha,c[2][0],int(c[2][1]),c[4]))
            consecutive_failures=0
        except Exception as e:
            for sha in batch: failures[sha]=repr(e)
            consecutive_failures+=1
        db.commit()
        if start%100==0 or start+10>=len(missing):
            print(f'Details attempted {min(start+10,len(missing))}/{len(missing)}; cached {db.execute("SELECT count(*) FROM commits").fetchone()[0]}; elapsed {time.monotonic()-started:.0f}s',flush=True)
        if consecutive_failures>=2:
            print('Stopping after two failed batches; all successful details are checkpointed.',flush=True)
            break
        time.sleep(2) # Ten commits per request, sequential requests with waits.
    data={r[0]:r[1:] for r in db.execute('SELECT sha,author,time,message FROM commits')}
    absent={p:[s for s in ss if s not in data] for p,ss in shas.items()}
    (CACHE/'missing_details.json').write_text(json.dumps(absent,indent=2))
    (CACHE/'retrieval_errors.json').write_text(json.dumps(failures,indent=2))
    if any(absent.values()):
        print('INCOMPLETE:',{p:len(ss) for p,ss in absent.items() if ss},flush=True)
        raise RuntimeError('Incomplete retrieval. Rerun to retry missing data; final CSV not replaced.')
    target=ROOT/(NETID+'_project_summary.csv')
    tmp=target.with_suffix('.csv.tmp')
    with tmp.open('w',encoding='utf-8',newline='') as f:
        w=csv.writer(f,delimiter=';'); w.writerow(FIELDS)
        for p,ss in shas.items():
            for s in sorted(ss,key=lambda s:(data[s][1],s)): w.writerow([p,s,*data[s]])
    tmp.replace(target)
    print('COMPLETE:',sum(map(len,shas.values())),'project-commit records',flush=True)


import json,time
from datetime import datetime,timezone

def collect():
    results=[]
    for p in projects():
        repo=p['GH'].split('github.com/')[1].rstrip('/')
        path=CACHE/(p['WoC']+'_github.json')
        if path.exists():
            result=json.loads(path.read_text(encoding='utf-8'))
        else:
            base='https://api.github.com/repos/'+repo
            detail=get_json(base)
            time.sleep(.5)
            recent=get_json(base+'/commits?per_page=10&until=2026-09-30T23%3A59%3A59Z')
            result={'project':p['WoC'],'url':p['GH'],'retrieved_at':datetime.now(timezone.utc).isoformat(),'repository':detail,'recent_commits_asof':recent,'cutoff':'2026-09-30T23:59:59Z'}
            path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
            time.sleep(.5)
        print(p['WoC'],result['repository']['stargazers_count'],result['repository']['forks_count'],result['recent_commits_asof'][0]['commit']['committer']['date'] if result['recent_commits_asof'] else 'NO COMMITS',flush=True)
        results.append(result)
    return results


from pathlib import Path
import os
import pandas as pd
from IPython.display import display, Image
ROOT = Path(os.environ.get('MP2_ROOT', Path.cwd()))
NETID = 'bdowlin2'
FIELDS = ['project_wocid','commit_sha1','author','time','commit message']
EXPECTED = {'avehtari_bda_course_aalto': 2272, 'eberharf_cfl': 1138, 'inqwire_sqir': 1338, 'jayggg_ngstents': 454, 'jgcri_gcam-core': 10677, 'openda-association_openda': 3202, 'openms_openms': 54828, 'tee-lab_pyddsde': 233, 'vatlab_sos': 8834, 'windweller_disextract': 559}
MANIFEST_EXPECTED = {'avehtari_bda_course_aalto': 2272, 'eberharf_cfl': 1138, 'openda-association_openda': 3209, 'openms_openms': 54864, 'vatlab_sos': 8834, 'jayggg_ngstents': 454, 'inqwire_sqir': 1338, 'tee-lab_pyddsde': 233, 'jgcri_gcam-core': 10688, 'windweller_disextract': 559}
OMITTED_SHA = {'avehtari_bda_course_aalto': [], 'eberharf_cfl': [], 'openda-association_openda': ['140121a8ec865bd6a0539c1618a5d2c142077e2d', '2a5674f7f773bf3f43496b4bf2e192de6770d9b7', '3683de7c2f1429beb45cac1921e3ccb5c82a3fa3', '44d0265855202b620b9ee3c9693e1ae03b680ef2', '6ea4b2ae8de635925c178f85262b69ae634a188a', '77c33ed192154c33d469abd3303fa75d8d1b4a7a', 'd137c7ad44156395ad6f27fb842c9528bce759a5'], 'openms_openms': ['129b8131f794cace10ad2ec3ce15042913a29581', '36999e78fe2d28e407a9f8f9e373a4fdbcea4add', '3ca10ef8b9dbaed2e620fcc886268495eb07fd70', '3db4de6ee30642be1ce2bbbe1f40c959bc399156', '4309527d3ba0e06baef2c61388bc4b37370cef1b', '4b7e74f50c55fd726b59de3e704a652475196137', '4c0a1c0bf4abe4e33119b6d509f5224130b85303', '520c2b0004518ff45988795b5335259359e23329', '5290369051952f1a445819649dcf60b98e459471', '55a7cbd29832b4308864f1dd88a21584f04df0fb', '5830392f5f774d8d8980d065395f37e8e77e6a21', '5af760a981893537f9c272cff13ca8f384533815', '5b5462d0aec8f9671e13ac5d6f0d4ab96765db99', '5c417fdcbe753cbf93a61d2c39b4890f35dbcecd', '6221cf80e1cfe2524b43bdf707578749070dfea6', '6ff496b5610c628486cc8065f49353875bd01807', '7b6d2d0d4a32ead2faa47ef1db097f8cdb7f807f', '7e3ba0839397164099c820d11ea5f7c60f82129a', '7ec2ea449b3da34f0a438e07de84bd5b2833e526', '7ec33827661b1e58cdaacb5ab2c4fdbbd5a4324d', '7f23ad8a236f406078e5d48746b61f13121b9dc9', '8152c0497a6bf9e62188102d5baebce67733907d', '815ce502bd97f548976d486a479817c0777318b6', '92a6a40e96358ff2f2ad63bc359e9adf29f1c813', '9d6ef6edcfd9c383017c4305093be04589cfce99', 'a1e0845cbe6125cbe93e79657ac9d0cecd06655c', 'a49e1d962709ae0f705d1ff93b2baeec2876497f', 'ae70bcc0ce460fff5735a8ac6f05d0ef2214a39c', 'b24e005a0e61fdde6ab798d4dba45353b7a92fa0', 'd6ef4de7603ae6be56a4a4f3cd01205f62d0bc15', 'd946a13a92b8fef08d5338243bf6328b1744ed79', 'e5d5cb5b30f988dc208409df79532879ce27dcf3', 'e8cc8894d6c71f4b747df9f3923c885cdbb39196', 'f59a014a80cca719a10f6d4ee0980b24672cc0a4', 'fb836abbe616ba1c7f1a686a3e267af63aa83b30', 'ff146b473baa0d9c57c265e4cb02cfd33f9f1373'], 'vatlab_sos': [], 'jayggg_ngstents': [], 'inqwire_sqir': [], 'tee-lab_pyddsde': [], 'jgcri_gcam-core': ['0932cec7bf6dcfb73b5b4959d4ff5acea8ffe839', '0f73ec81b518038c30084f53dd0828bda4dd323d', '2c18636e69b27ebac8950fe16ce9e25fa0685a49', '3214af619327a9d1f77b10087f56e368adb7b3ba', '32b0b0e6253f3bc41129209ef8bf338cf1a8bbda', '6c4823bd3d4bb1ef5d8348c92d94860c7fabe40e', '70ec8bb344c7a849abf76fc477445dcc5a7c3611', '7565d844d2542fe8b54dc2a1ec9319397bd01904', '7cbcf67c0b0bf2506ff7e707556c296aa34b6c8d', '93ba023ab030d5e4a8a51360e971c52c1aeb5737', 'd73e681e97086a076aa3e4a3b880f6b36f1cc95a'], 'windweller_disextract': []}

def read_submission():
    data = pd.read_csv(ROOT / 'bdowlin2_project_summary.csv', sep=';', keep_default_na=False,
                       dtype={'project_wocid':str,'commit_sha1':str,'author':str,'time':'int64','commit message':str})
    assert list(data.columns) == FIELDS
    assert not data.duplicated(['project_wocid','commit_sha1']).any()
    assert data.author.ne('').all() and data.commit_sha1.str.fullmatch('[0-9a-f]{40}').all()
    assert set(data.project_wocid) == set(EXPECTED)
    for name, part in data.groupby('project_wocid'):
        assert len(part) == EXPECTED[name], f'Unexpected analyzed count: {name}'
        assert EXPECTED[name] + len(OMITTED_SHA[name]) == MANIFEST_EXPECTED[name]
        assert not set(part.commit_sha1) & set(OMITTED_SHA[name])
    data['date'] = pd.to_datetime(data.time, unit='s', utc=True)
    return data

assigned = pd.DataFrame(projects())
display(assigned)

,netID,WoC,GH
0,bdowlin2,avehtari_bda_course_aalto,https://github.com/avehtari/BDA_course_Aalto
1,bdowlin2,eberharf_cfl,https://github.com/eberharf/cfl
2,bdowlin2,openda-association_openda,https://github.com/OpenDA-Association/OpenDA
3,bdowlin2,openms_openms,https://github.com/OpenMS/OpenMS
4,bdowlin2,vatlab_sos,https://github.com/vatlab/SoS
5,bdowlin2,jayggg_ngstents,https://github.com/jayggg/ngstents
6,bdowlin2,inqwire_sqir,https://github.com/inQWIRE/SQIR
7,bdowlin2,tee-lab_pyddsde,https://github.com/tee-lab/pyddsde
8,bdowlin2,jgcri_gcam-core,https://github.com/JGCRI/gcam-core
9,bdowlin2,windweller_disextract,https://github.com/windweller/DisExtract


## Retrieval code and submitted dataset

The code above records the original resumable WoC retrieval method: ten commit objects per sequential request, waits, bounded retries, and a SQLite checkpoint. Collection is now closed under the TA-approved reduced-data approach; keep `RUN_NETWORK = False` to review this final dataset without starting new collection. The original retrieval function requires full coverage before exporting, so the final CSV was exported separately after validating that only the 54 documented unavailable hashes are omitted.

This notebook runs offline from the submitted summary CSV and `net2prj.csv`, without local helper modules or pre-existing caches. Expected analyzed counts and the exact omission lists are embedded below. Before export, the submitted SHA sets were checked against the complete WoC manifests minus exactly those omission lists.

In [2]:
RUN_NETWORK = False
if RUN_NETWORK:
    collect()
    retrieve()
else:
    print('Offline submission review; no network retrieval started.')

Offline submission review; no network retrieval started.


In [3]:
summary = read_submission()
submitted = summary.groupby('project_wocid').size().to_dict()
coverage = pd.DataFrame([{'Project':name, 'WoC manifest records':MANIFEST_EXPECTED[name],
                         'Analyzed records':submitted[name], 'Unavailable objects':len(OMITTED_SHA[name]),
                         'WoCDataComplete':not OMITTED_SHA[name], 'AnalysisReady':submitted[name]==EXPECTED[name]}
                        for name in EXPECTED])
display(coverage)
assert coverage['AnalysisReady'].all()
print('Analyzed records:',len(summary),'across all ten assigned projects; unavailable objects:',sum(map(len,OMITTED_SHA.values())))

,Project,WoC manifest records,Analyzed records,Unavailable objects,WoCDataComplete,AnalysisReady
0,avehtari_bda_course_aalto,2272,2272,0,True,True
1,eberharf_cfl,1138,1138,0,True,True
2,inqwire_sqir,1338,1338,0,True,True
3,jayggg_ngstents,454,454,0,True,True
4,jgcri_gcam-core,10688,10677,11,False,True
5,openda-association_openda,3209,3202,7,False,True
6,openms_openms,54864,54828,36,False,True
7,tee-lab_pyddsde,233,233,0,True,True
8,vatlab_sos,8834,8834,0,True,True
9,windweller_disextract,559,559,0,True,True


Analyzed records: 83535 across all ten assigned projects; unavailable objects: 54


## Required project statistics

The text cells report GitHub metadata and metrics for the retrieved WoC records. Omission counts are explicit; raw author strings are not deduplicated people.

In [4]:
woc_stats = summary.groupby('project_wocid').agg(ncommits=('commit_sha1','nunique'), nauthors=('author','nunique'), min_time=('date','min'), max_time=('date','max'))
display(woc_stats)

,ncommits,nauthors,min_time,max_time
project_wocid,,,,
avehtari_bda_course_aalto,2272,67,2018-09-07 14:11:11+00:00,2025-11-03 15:40:44+00:00
eberharf_cfl,1138,14,2020-07-17 21:53:13+00:00,2025-05-07 22:32:37+00:00
inqwire_sqir,1338,34,2018-12-20 22:11:21+00:00,2025-07-30 02:27:23+00:00
jayggg_ngstents,454,11,2020-06-29 20:41:15+00:00,2025-06-10 12:50:41+00:00
jgcri_gcam-core,10677,161,2002-10-14 17:48:16+00:00,2026-04-01 20:36:00+00:00
openda-association_openda,3202,63,2014-09-24 12:58:34+00:00,2025-10-22 13:02:45+00:00
openms_openms,54828,538,2006-06-11 05:48:53+00:00,2026-04-17 10:04:15+00:00
tee-lab_pyddsde,233,7,2020-03-30 14:40:44+00:00,2021-10-29 06:08:17+00:00
vatlab_sos,8834,55,2016-02-13 00:46:46+00:00,2025-11-06 14:29:35+00:00


### avehtari_bda_course_aalto

[GitHub repository](https://github.com/avehtari/BDA_course_Aalto)

- GitHub stars: **2282**; forks: **566** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-29T11:32:50Z**.
- WoC commit count: **2,272** distinct hashes.
- Analyzed WoC commits: **2,272** of **2,272** manifest records; **0** unavailable objects.
- Observed author strings: **67**; minimum time: **2018-09-07T14:11:11+00:00**; maximum time: **2025-11-03T15:40:44+00:00**.

### eberharf_cfl

[GitHub repository](https://github.com/eberharf/cfl)

- GitHub stars: **19**; forks: **1** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-05-05T21:52:43Z**.
- WoC commit count: **1,138** distinct hashes.
- Analyzed WoC commits: **1,138** of **1,138** manifest records; **0** unavailable objects.
- Observed author strings: **14**; minimum time: **2020-07-17T21:53:13+00:00**; maximum time: **2025-05-07T22:32:37+00:00**.

### openda-association_openda

[GitHub repository](https://github.com/OpenDA-Association/OpenDA)

- GitHub stars: **104**; forks: **34** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-07-10T13:57:20Z**.
- WoC commit count: **3,209** distinct hashes.
- Analyzed WoC commits: **3,202** of **3,209** manifest records; **7** unavailable objects.
- Observed author strings: **63**; minimum time: **2014-09-24T12:58:34+00:00**; maximum time: **2025-10-22T13:02:45+00:00**.

### openms_openms

[GitHub repository](https://github.com/OpenMS/OpenMS)

- GitHub stars: **623**; forks: **445** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-30T17:00:08Z**.
- WoC commit count: **54,864** distinct hashes.
- Analyzed WoC commits: **54,828** of **54,864** manifest records; **36** unavailable objects.
- Observed author strings: **538**; minimum time: **2006-06-11T05:48:53+00:00**; maximum time: **2026-04-17T10:04:15+00:00**.

### vatlab_sos

[GitHub repository](https://github.com/vatlab/SoS)

- GitHub stars: **286**; forks: **44** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-09T18:36:17Z**.
- WoC commit count: **8,834** distinct hashes.
- Analyzed WoC commits: **8,834** of **8,834** manifest records; **0** unavailable objects.
- Observed author strings: **55**; minimum time: **2016-02-13T00:46:46+00:00**; maximum time: **2025-11-06T14:29:35+00:00**.

### jayggg_ngstents

[GitHub repository](https://github.com/jayggg/ngstents)

- GitHub stars: **9**; forks: **6** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-11-27T00:17:25Z**.
- WoC commit count: **454** distinct hashes.
- Analyzed WoC commits: **454** of **454** manifest records; **0** unavailable objects.
- Observed author strings: **11**; minimum time: **2020-06-29T20:41:15+00:00**; maximum time: **2025-06-10T12:50:41+00:00**.

### inqwire_sqir

[GitHub repository](https://github.com/inQWIRE/SQIR)

- GitHub stars: **105**; forks: **26** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-07-30T02:27:23Z**.
- WoC commit count: **1,338** distinct hashes.
- Analyzed WoC commits: **1,338** of **1,338** manifest records; **0** unavailable objects.
- Observed author strings: **34**; minimum time: **2018-12-20T22:11:21+00:00**; maximum time: **2025-07-30T02:27:23+00:00**.

### tee-lab_pyddsde

[GitHub repository](https://github.com/tee-lab/pyddsde)

- GitHub stars: **98**; forks: **12** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-06-10T04:11:37Z**.
- WoC commit count: **233** distinct hashes.
- Analyzed WoC commits: **233** of **233** manifest records; **0** unavailable objects.
- Observed author strings: **7**; minimum time: **2020-03-30T14:40:44+00:00**; maximum time: **2021-10-29T06:08:17+00:00**.

### jgcri_gcam-core

[GitHub repository](https://github.com/JGCRI/gcam-core)

- GitHub stars: **432**; forks: **226** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-06-01T21:04:27Z**.
- WoC commit count: **10,688** distinct hashes.
- Analyzed WoC commits: **10,677** of **10,688** manifest records; **11** unavailable objects.
- Observed author strings: **161**; minimum time: **2002-10-14T17:48:16+00:00**; maximum time: **2026-04-01T20:36:00+00:00**.

### windweller_disextract

[GitHub repository](https://github.com/windweller/DisExtract)

- GitHub stars: **0**; forks: **0** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-07-13T15:34:42Z**.
- WoC commit count: **559** distinct hashes.
- Analyzed WoC commits: **559** of **559** manifest records; **0** unavailable objects.
- Observed author strings: **9**; minimum time: **2017-10-29T23:56:48+00:00**; maximum time: **2025-07-13T15:34:42+00:00**.

## Validation and limitations

The final 83,535-record CSV contains all ten projects. Export validation compared exact SHA sets against each WoC manifest minus exactly the 54 authorized omissions, with no unexpected missing or extra records. Offline checks validate schema, unique project/SHA keys, SHA format, author fields, timestamps, analyzed counts, and exclusion of every documented missing object.

`bdowlin2_source_comparison.csv` compares WoC manifest counts with GitHub default-branch counts; their history scopes differ. The final statistics use analyzed counts, with separate expected and omitted columns. For affected projects, absent timestamps/messages could alter date extrema, author totals, gaps, or sample membership; conclusions are observations from available records. This limitation is accepted under the TA's reduced-record guidance and is retained in the submission documentation.